# Research on static data correlation and so on.

In [2]:
%load_ext autoreload
%autoreload 2

In [8]:
import os

import polars as pl
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

from utils.datasets import HydroStaticFeaturesFiles
from utils.datasets import MeteoSeriesFeaturesFiles
from utils.datasets import HydroFiles
from utils.datasets import ROOT_FOLDER, ARTIFACTS_FOLDER

import pickle as pkl


os.chdir(ROOT_FOLDER)

## Plan (correlation analyzis)

- Load data
    - select where not nulls
- Evaluate pairwise correlation
    - for all data
    - based on clusters
- From the list of variables with high correlation $\left(\mid \cdot \mid > 0.8 \right)$, select the most meaningful ones. Optionally, describe possible reasons why the data might be correlated.
- Make subset.

In [40]:
hsff_lf = HydroStaticFeaturesFiles().dataframe

In [7]:
def compute_corr_matrix(
    lf: pl.LazyFrame,
    method: str = "pearson"
) -> pd.DataFrame:
    numeric_columns = lf.select(pl.selectors.numeric()).collect_schema().names()
    corr_dict = {
        col1: [
            lf.select(
                pl.corr(pl.col(col1), pl.col(col2), method=method)
            ).collect().item()
            for col2 in numeric_columns
        ]
        for col1 in tqdm(numeric_columns)
    }

    corr_df = pd.DataFrame(corr_dict, index=numeric_columns)
    return corr_df


corr_matrix = compute_corr_matrix(hsff_lf)

  0%|          | 0/150 [00:00<?, ?it/s]

In [9]:
with open(ARTIFACTS_FOLDER / "hsff_corr_matrix.pkl", "wb") as file:
    pkl.dump(corr_matrix, file)

In [14]:
preproc_corr_matrix = corr_matrix.fillna(1).abs()

# upper_triangle = preproc_corr_matrix.where(
#     np.triu(np.ones(preproc_corr_matrix.shape), k=1).astype(bool)
# )

# high_corr_pairs = upper_triangle.stack()[lambda x: x > 0.8].sort_values(ascending=False)

# for (feature1, feature2), corr in high_corr_pairs.items():
#     print(f"{feature1} — {feature2}: {corr:.2f}")

In [19]:
clustermap = sns.clustermap(
    preproc_corr_matrix,
    cmap='coolwarm',
    figsize=(40, 40)
)

clustermap.fig.suptitle("Correlation Heatmap", fontsize=24)

# clustermap.fig.subplots_adjust(top=0.95)
clustermap.fig.tight_layout()

clustermap.fig.savefig(ARTIFACTS_FOLDER / "static_features_heatmap.pdf", format='pdf', bbox_inches='tight')

plt.close(clustermap.fig)

In [24]:
def select_uncorrelated_features(
    corr_matrix: pd.DataFrame,
    feature_scores: pd.Series,
    threshold: float = 0.8,
    method: str = 'average'
):
    """
    Selects the most informative feature per correlation cluster.

    Parameters:
    - corr_matrix: pd.DataFrame — symmetric correlation matrix
    - feature_scores: pd.Series — informativeness score per feature (e.g., variance, model importance)
    - threshold: float — absolute correlation threshold (e.g., 0.8)
    - method: str — linkage method for clustering

    Returns:
    - List of selected feature names
    - DataFrame mapping all features to their cluster and scores
    """
    import numpy as np
    from scipy.spatial.distance import squareform
    from scipy.cluster.hierarchy import linkage, fcluster
    import pandas as pd

    # Validate input
    assert isinstance(corr_matrix, pd.DataFrame)
    assert isinstance(feature_scores, pd.Series)
    assert set(corr_matrix.columns) == set(feature_scores.index), "Mismatch in features"

    # Convert to distance matrix
    dist_matrix = 1 - np.abs(corr_matrix)
    condensed_dist = squareform(dist_matrix, checks=False)

    # Hierarchical clustering
    Z = linkage(condensed_dist, method=method)
    cluster_labels = fcluster(Z, t=1 - threshold, criterion='distance')

    # Cluster assignment
    features = corr_matrix.columns
    df = pd.DataFrame({
        'feature': features,
        'cluster': cluster_labels,
        'score': feature_scores[features].values
    })

    # Select top feature per cluster based on score
    representatives = df.sort_values('score', ascending=False).groupby('cluster').first().reset_index()
    selected_features = representatives['feature'].tolist()

    return selected_features, df

In [39]:
feature_scores = hsff_lf.var().collect().to_pandas().iloc[0]

selected_features, cluster_df = select_uncorrelated_features(
    corr_matrix=preproc_corr_matrix.clip(upper=1),
    feature_scores=feature_scores,
    threshold=0.8
)

print(f"Selected {len(selected_features)} features:")
print(selected_features)

Selected 56 features:
['rev_mc_usu', 'dor_pc_pva', 'wet_pc_s02', 'urb_pc_sse', 'gla_pc_sse', 'height_bs', 'acc', 'wet_pc_s07', 'lkv_mc_usu', 'wet_pc_s08', 'glc_pc_s13', 'wet_pc_s09', 'kar_pc_sse', 'glc_pc_s06', 'ero_kh_sav', 'ele_mt_sav', 'pre_mm_s07', 'pre_mm_s06', 'aet_mm_s09', 'lon', 'aet_mm_s06', 'swc_pc_s12', 'glc_pc_s14', 'glc_pc_s09', 'glc_pc_s15', 'snw_pc_s03', 'tmp_dc_s04', 'cmi_ix_s07', 'pet_mm_s11', 'pet_mm_s02', 'tmp_dc_s11', 'aet_mm_s05', 'prm_pc_sse', 'cmi_ix_s03', 'glc_pc_s17', 'snw_pc_s05', 'snw_pc_s06', 'snd_pc_sav', 'glc_pc_s04', 'hft_ix_s09', 'snw_pc_s04', 'soc_th_sav', 'ire_pc_sse', 'for_pc_sse', 'pst_pc_sse', 'pet_mm_s05', 'pet_mm_s06', 'cmi_ix_s10', 'cmi_ix_s11', 'inu_pc_ult', 'gauge_id', 'cmi_ix_s08', 'glc_pc_s10', 'glc_pc_s02', 'wet_pc_s04', 'wet_pc_s03']
